# 01 — Data Filtering & Cleaning
Loads the raw HealthCareMagic-100k dataset and filters it down
to high-quality, relevant medical QA samples for fine-tuning Swasthya.

**Input:** HealthCareMagic-100k (112,165 samples)  
**Output:** `cleaned_data.json` (2,567 samples)


## Loading Raw Dataset

In [ ]:
from datasets import load_dataset

ds=load_dataset("Malikeh1375/medical-question-answering-datasets",
                  "chatdoctor_healthcaremagic")

print(ds)
print(ds['train'][0])

In [ ]:
type(ds)

datasets.dataset_dict.DatasetDict

In [ ]:
ds.shape

{'train': (112165, 3)}

## Inspecting Raw Samples

In [ ]:
sample = ds['train'][0]
print("INSTRUCTION:", sample['instruction'])
print("\nINPUT:", sample['input'])
print("\nOUTPUT:", sample['output'])

INSTRUCTION: If you are a doctor, please answer the medical questions based on the patient's description.

INPUT: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!

OUTPUT: Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomitin

In [ ]:
for i in range(10):
    sample = ds['train'][i]
    print(f"--- Sample {i+1} ---")
    print("INPUT:", sample['input'][:150])
    print("OUTPUT:", sample['output'][:200])
    print()

--- Sample 1 ---
INPUT: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i fe
OUTPUT: Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom i

--- Sample 2 ---
INPUT: My baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are very watery with green stringy bits in the
OUTPUT: Hi... Thank you for consulting in Chat Doctor. It seems your kid is having viral diarrhea. Once it starts it will take 5-7 days to completely get better. Unless the kids having low urine output or ver

--- Sample 3 ---
INPUT: Hello, My husband is taking Oxycodone due to a broken leg/surgery. He has been taking this pain medication for one month. We are trying to conceive ou
OUTPUT: Hello, and I hope I can help you today.First, t

## Filter Rules & Safety Keywords

In [ ]:
import re

symptoms = [
    # Physical symptoms
    "fever", "cold", "cough", "sore throat", "headache",
    "dizziness", "nausea", "vomiting", "diarrhea", "stomach",
    "rash", "burn", "cut", "wound", "fatigue", "body ache",
    "congestion", "flu", "runny nose", "chills", "weakness",
    "eye", "ear", "pain", "swelling", "itching", "allergy",
    "constipation", "bloating", "indigestion", "heartburn",
    "back pain", "joint pain", "muscle pain", "cramp",
    "bleeding", "bruise", "sprain", "fracture", "infection",
    "skin", "acne", "eczema", "dry skin", "hair loss",
    "chest pain", "shortness of breath", "palpitation",
    "urination", "kidney", "liver", "thyroid",

    # Mental health (mild)
    "anxiety", "stress", "depression", "sleep",
    "insomnia", "mood", "nervous", "panic",
    "tired", "exhausted", "mental", "worry",
    "overthinking", "restless", "irritable"
]

exclude_medical = [
    "cancer", "tumor", "biopsy", "biopsied", "lump",
    "chemotherapy", "radiation",
    "hiv", "aids", "hepatitis", "tuberculosis", "tb",
    "diabetes", "insulin", "dialysis",
    "pregnancy", "pregnant", "trimester", "labour", "delivery",
    "oxycodone", "morphine", "cocaine", "heroin", "narcotic",
    "epilepsy", "seizure", "stroke", "paralysis",
    "ild", "interstitial", "degenerative", "osteoporosis",
    "schizophrenia", "bipolar", "psychosis", "hallucination",
    "erectile", "sexually transmitted", "std", "sti",
    "surgery", "operation", "transplant", "amputation",
    "gynecological", "ovarian", "uterus", "endometriosis",
    "emergency room", "icu", "life threatening",
    "critical condition", "immediately rush",
    "alt", "ast", "liver function", "blood test", "lab report",
    "reading came back", "test result", "blood work",
    "cholesterol", "triglyceride", "glucose level",
    "blood pressure", "bp reading", "ecg", "mri", "ct scan",
    "ultrasound", "x-ray", "report shows",
    "antibiotic", "prescribed", "prescription", "medication",
    "tablet", "capsule", "mg ", "dosage", "dose",
    "bupropion", "antidepressant", "paracetamol",
    "ibuprofen", "amoxicillin", "azithromycin",
    "sgpt", "bilirubin", "fatty liver", "trichophagia",
    "braces", "orthodontic", "aligner", "impulse control"
]

# Mental health crisis
crisis_keywords = [
    "suicide", "suicidal", "kill myself", "end my life",
    "self harm", "self-harm", "cut myself", "hurt myself",
    "want to die", "no reason to live", "overdose on purpose"
]

# Boilerplate to remove from outputs
boilerplate_patterns = [
    r"hi,?\s*thank you for (posting|consulting|using chat doctor)[^.]*\.",
    r"thank you for (posting|consulting|using chat doctor)[^.]*\.",
    r"welcome to chat doctor[^.]*\.",
    r"i am chat doctor[^.]*\.",
    r"dear friend[^.]*\.",
    r"hello,?\s*and i hope i can help you today[^.]*\.",
    r"chat doctor\.?$",
    r"best wishes,?\s*chat doctor\.?",
    r"hope this helps\.?\s*chat doctor\.?",
    r"take care\.?\s*chat doctor\.?",
    r"thank you for (asking|writing to|contacting) chat doctor[^.]*\.",
    r"i have read your report[^.]*\.",
    r"i have gone through your (report|query|question)[^.]*\.",
    r"chat doctor\.?"
]

CRISIS_RESPONSE = (
    "আমি বুঝতে পারছি আপনি এখন অনেক কঠিন সময়ের মধ্যে আছেন। "
    "অনুগ্রহ করে এখনই একজন বিশেষজ্ঞের সাথে কথা বলুন। "
    "iCall হেল্পলাইন: 9152987821। "
    "আপনি একা নন — সাহায্য পাওয়া সম্ভব।"
)


## Output Quality Checks

In [ ]:
import re

def is_quality_output(output):

    # Must have minimum word count — at least 20 words
    if len(output.split()) < 20:
        return False

    # Must have maximum word count — not too long
    if len(output.split()) > 200:
        return False

    # No URLs or email addresses
    if re.search(r'http|www\.|@gmail|@yahoo', output.lower()):
        return False

    incomplete_endings = [
    "following reasons", "as follows",
    "following:", "such as", "including"
    ]
    if any(output.lower().endswith(e) for e in incomplete_endings):
        return False

    return True

## Cleaning Function & Filter Function

In [ ]:
# CLEANING FUNCTION

def clean_output(text):
    text = text.strip()
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()
    # Remove multiple spaces and newlines
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# FILTER FUNCTION

def is_relevant(example):
    input_text  = example['input'].lower()
    output_text = example['output'].lower()

    # Always exclude crisis content

    if any(word in input_text for word in crisis_keywords):
        return False

    # Exclude too serious/complex medical content
    if any(word in input_text or word in output_text
           for word in exclude_medical):
        return False

    # Must contain at least one relevant symptom
    has_symptom = any(s in input_text for s in symptoms)
    if not has_symptom:
        return False

    # Remove very short inputs — not enough context
    if len(example['input']) < 100:
        return False

    # Remove very long outputs — too complex
    if len(example['output']) > 1000:
        return False

    cleaned_output = clean_output(example['output'])
    if not is_quality_output(cleaned_output):
        return False

    return True

## Applying the Filter

In [ ]:
# APPLY FILTER

filtered_ds = ds['train'].filter(is_relevant)
print(f"Original : {len(ds['train']):,} rows")
print(f"Filtered : {len(filtered_ds):,} rows")

Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

Original : 112,165 rows
Filtered : 2,567 rows


## Cleaning the Filtered Data

In [ ]:
def process_example(example):
    input_text  = example['input'].strip()
    output_text = clean_output(example['output'])

    # Check if crisis — add helpline response
    if any(word in input_text.lower() for word in crisis_keywords):
        output_text = CRISIS_RESPONSE

    return {
        "input"  : input_text,
        "output" : output_text
    }

cleaned_ds = filtered_ds.map(process_example)

# Quick check
print("Sample after cleaning:")
print("INPUT :", cleaned_ds[0]['input'][:200])
print("OUTPUT:", cleaned_ds[0]['output'][:200])
print(f"\nTotal clean rows: {len(cleaned_ds):,}")

Map:   0%|          | 0/2567 [00:00<?, ? examples/s]

Sample after cleaning:
INPUT : I have noticed that I ve been getting these sore reddish spots with a white ring around them on my tounge.the spot on my tounge is reddish with a white ring around it and the other 3 are bigger patche
OUTPUT: Hello there, You may have a manifestation of a micronutrient deficiency. You should have a multivitamin once a day for the next 1 week preferably one with vitamin b-12(cyanocobalamine) and then assess

Total clean rows: 2,567


## Final Spot Check

In [ ]:
import random
random.seed(99)

indices = random.sample(range(len(cleaned_ds)), 5)

for i, idx in enumerate(indices):
    sample = cleaned_ds[idx]
    print(f"--- Sample {i+1} ---")
    print("INPUT :", sample['input'][:200])
    print("OUTPUT:", sample['output'][:300])
    print()

--- Sample 1 ---
INPUT : I am a 73 year old woman with muscle ache in shoulders, arm and back of one hand. I have diagnosed arthritis in my neck. I have also had previous ulnar issues. Is this being caused by my neck or some 
OUTPUT: Hi, Pain in your shoulders, arms and back of the hands might be related to arthritic changes in your neck. These changes cause pain from nerve compression and irritation and radiate in arms and hands. The treatment is the same, no matter if the cause is arthritis of the neck or ulnar issue. You shou

--- Sample 2 ---
INPUT : My four year old granddaughter has been complaining of leg pain...her pediatrician thinks it s growing pains, but she complains alot and says it hurts to walk. Just this morning we noticed a very dark
OUTPUT: Hi, sorry to hear about your granddaughter leg pain. As your pediatrician suggested leg pain is common during toddler age groups but they don't feel it much during daytime. They will complain about it more at night. Regarding the 

## Save to Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os

# Create save directory
save_dir = '/content/drive/MyDrive/swasthya/data'
os.makedirs(save_dir, exist_ok=True)

# Convert to list of dicts and save
data = []
for row in cleaned_ds:
    data.append({
        'input': row['input'],
        'output': row['output']
    })

with open(f'{save_dir}/cleaned_data.json', 'w',
          encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(data)} samples to Google Drive")

Mounted at /content/drive
Saved 2567 samples to Google Drive
